## Example 1: PublicClientApplication and basic authentication setup

In [ ]:
import base64
import json
import urllib.parse

import msal

class FakeAuthorityResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.text = json.dumps(payload)
        self.headers = {"Content-Type": "application/json"}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")

class FakeAuthority:
    """Answers MSAL the way login.microsoftonline.com does, from a script of replies, and records every request."""
    HOST = "https://login.microsoftonline.com"

    def __init__(self, *replies):
        self.requests, self.replies = [], list(replies)

    def get(self, url, params=None, headers=None, **kwargs):
        self.requests.append(("GET", url, params, None))
        path = urllib.parse.urlsplit(url).path
        if path == "/common/discovery/instance":
            return FakeAuthorityResponse(200, {"api-version": "1.1", "metadata": [{
                "preferred_network": "login.microsoftonline.com", "preferred_cache": "login.windows.net",
                "aliases": ["login.microsoftonline.com", "login.windows.net", "login.microsoft.com", "sts.windows.net"]}]})
        assert path.endswith("/v2.0/.well-known/openid-configuration"), f"MSAL asked for an unexpected GET {url}"
        tenant = path.strip("/").split("/")[0]
        endpoints = f"{self.HOST}/{tenant}/oauth2/v2.0"
        return FakeAuthorityResponse(200, {
            "issuer": f"{self.HOST}/{tenant}/v2.0", "authorization_endpoint": f"{endpoints}/authorize",
            "token_endpoint": f"{endpoints}/token", "device_authorization_endpoint": f"{endpoints}/devicecode"})

    def post(self, url, params=None, data=None, headers=None, **kwargs):
        self.requests.append(("POST", url, params, data))
        assert self.replies, f"MSAL made a POST the example did not script: {url}"
        return FakeAuthorityResponse(*self.replies.pop(0))

def b64url_json(payload):
    return base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip("=")

# Example 1: PublicClientApplication and basic authentication setup
print("msal PublicClientApplication and authentication setup")

try:
    print("Example 1 - PublicClientApplication and basic authentication setup")
    
    # Create a public client application
    # Note: These are demo values - real applications need proper configuration
    client_id = "your-client-id-here"  # Would be actual Azure AD app registration client ID
    authority = "https://login.microsoftonline.com/common"  # Microsoft common authority
    fake_authority = FakeAuthority()
    
    # Initialize PublicClientApplication
    app = msal.PublicClientApplication(
        client_id=client_id,
        authority=authority,
        # In production, you might want to specify a custom cache
        # token_cache=token_cache
        http_client=fake_authority,
    )
    # Construction resolves the authority: exactly one OpenID discovery GET, for the configured tenant.
    assert [request[1] for request in fake_authority.requests] == [
        "https://login.microsoftonline.com/common/v2.0/.well-known/openid-configuration"
    ], fake_authority.requests
    
    print(f"PublicClientApplication created:")
    print(f"  Client ID: {client_id[:8]}...")
    print(f"  Authority: {authority}")
    print(f"  Application type: {type(app).__name__}")
    
    # Define scopes for Microsoft Graph
    scopes = [
        "User.Read",           # Read user profile
        "Mail.Read",           # Read user's mail
        "Calendars.Read"       # Read user's calendar
    ]
    
    print(f"Configured scopes:")
    for scope in scopes:
        print(f"    {scope}")
    
    # Get accounts: a new app starts with an empty token cache
    accounts = app.get_accounts()
    assert accounts == [], accounts
    print(f"Cached accounts: {len(accounts)} found")
    
    # Sign in with the authorization code flow + PKCE. MSAL builds the URL the user visits; `form_post` has the
    # authority POST the code back instead of putting it in the URL (MSAL's own browser flow requires it).
    flow = app.initiate_auth_code_flow(scopes, redirect_uri="http://localhost:8400", response_mode="form_post")
    auth_query = urllib.parse.parse_qs(urllib.parse.urlsplit(flow["auth_uri"]).query)
    assert flow["auth_uri"].startswith("https://login.microsoftonline.com/common/oauth2/v2.0/authorize?"), flow["auth_uri"]
    assert auth_query["client_id"] == [client_id] and auth_query["response_type"] == ["code"], auth_query
    assert auth_query["response_mode"] == ["form_post"] and auth_query["code_challenge_method"] == ["S256"], auth_query
    assert auth_query["state"] == [flow["state"]] and auth_query["redirect_uri"] == ["http://localhost:8400"], auth_query
    assert set(auth_query["scope"][0].split()) == set(scopes) | {"offline_access", "openid", "profile"}, auth_query
    print(f"Sign-in URL built: {flow['auth_uri'][:60]}...")
    
    # The authority's answer to the code, as the real endpoint gives it: tokens plus an ID token for the signed-in user
    # that echoes the nonce MSAL put in the URL (MSAL refuses a mismatch — replay protection).
    fake_authority.replies.append((200, {
        "token_type": "Bearer", "scope": "User.Read Mail.Read Calendars.Read openid profile", "expires_in": 3599,
        "access_token": "eyJ0eXAi.fake-user-access-token", "refresh_token": "0.fake-refresh-token",
        "client_info": b64url_json({"uid": "00000000-0000-0000-0000-00000000abcd", "utid": "72f988bf-86f1-41af-91ab-2d7cd011db47"}),
        "id_token": b64url_json({"alg": "none", "typ": "JWT"}) + "." + b64url_json({
            "aud": client_id, "iss": "https://login.microsoftonline.com/72f988bf-86f1-41af-91ab-2d7cd011db47/v2.0",
            "tid": "72f988bf-86f1-41af-91ab-2d7cd011db47", "oid": "00000000-0000-0000-0000-00000000abcd",
            "name": "John Doe", "preferred_username": "john@example.com", "nonce": auth_query["nonce"][0]}) + "."}))
    
    # An auth response that does not carry this flow's state is refused before any token request is made.
    try:
        app.acquire_token_by_auth_code_flow(dict(flow), {"code": "fake-auth-code", "state": "forged-state"})
    except ValueError as refusal:
        assert "state mismatch" in str(refusal), refusal
    else:
        raise AssertionError("MSAL redeemed an auth response whose state was forged")
    
    token_response = app.acquire_token_by_auth_code_flow(flow, {"code": "fake-auth-code", "state": flow["state"]})
    assert token_response["access_token"] == "eyJ0eXAi.fake-user-access-token", token_response
    assert token_response["token_type"] == "Bearer" and token_response["token_source"] == "identity_provider", token_response
    assert token_response["id_token_claims"]["preferred_username"] == "john@example.com", token_response
    # The request MSAL formed: one authorization-code POST to the tenant's token endpoint, carrying the PKCE verifier.
    token_posts = [request for request in fake_authority.requests if request[0] == "POST"]
    assert len(token_posts) == 1, fake_authority.requests
    _, token_url, _, token_form = token_posts[0]
    assert token_url == "https://login.microsoftonline.com/common/oauth2/v2.0/token", token_url
    assert token_form["grant_type"] == "authorization_code" and token_form["code"] == "fake-auth-code", token_form
    assert token_form["code_verifier"] == flow["code_verifier"], token_form
    assert token_form["redirect_uri"] == "http://localhost:8400" and token_form["client_id"] == client_id, token_form
    
    # The signed-in account is now cached, and a silent call is served from the cache with no request at all.
    accounts = app.get_accounts()
    assert [account["username"] for account in accounts] == ["john@example.com"], accounts
    requests_before_silent = len(fake_authority.requests)
    silent_result = app.acquire_token_silent(["User.Read"], account=accounts[0])
    assert silent_result["access_token"] == token_response["access_token"], silent_result
    assert silent_result["token_source"] == "cache", silent_result
    assert len(fake_authority.requests) == requests_before_silent, fake_authority.requests
    
    print(f"Example account structure:")
    for key, value in accounts[0].items():
        print(f"  {key}: {value}")
    
    # Authentication flow options
    auth_flows = {
        "Interactive": "acquire_token_interactive() - Opens browser for user login",
        "Silent": "acquire_token_silent() - Uses cached tokens if available",
        "Device Code": "initiate_device_flow() + acquire_token_by_device_flow() - For devices without browsers",
        "Username/Password": "acquire_token_by_username_password() - Not recommended",
        "Authorization Code": "initiate_auth_code_flow() + acquire_token_by_auth_code_flow() - Web app flow"
    }
    
    print(f"Available authentication flows:")
    for flow_name, description in auth_flows.items():
        print(f"  • {flow_name}: {description}")
    
    # Token structure: the response MSAL returned above (secrets shortened)
    print(f"Token response structure:")
    for key, value in token_response.items():
        if key in ("access_token", "refresh_token", "id_token", "client_info"):
            print(f"  {key}: {str(value)[:20]}...")
        else:
            print(f"  {key}: {value}")
    
    # MSAL configuration options
    print(f"MSAL configuration options:")
    
    config_options = [
        "authority: Identity provider endpoint (Azure AD, B2C, ADFS)",
        "validate_authority: Verify authority is valid (default: True)",
        "cache_location: Token cache storage location",
        "http_client: Custom HTTP client for network requests",
        "proxies: HTTP proxy configuration",
        "timeout: Request timeout settings",
        "instance_discovery: Automatic instance discovery (default: True)"
    ]
    
    for option in config_options:
        print(f"  • {option}")
    
    # Security best practices
    print(f"Security best practices:")
    
    security_practices = [
        "Use HTTPS only: Never transmit tokens over unencrypted connections",
        "Secure token storage: Store refresh tokens securely, never in plain text",
        "Scope principle: Request minimum necessary scopes",
        "Token validation: Always validate tokens before use",
        "Expiry handling: Check token expiration and refresh automatically",
        "Failure handling: Check every MSAL result for an 'error' key before using a token"
    ]
    
    for practice in security_practices:
        print(f"  • {practice}")
    
    print("msal example 1 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: ConfidentialClientApplication and token cache management

In [ ]:
import base64
import datetime
import json
import urllib.parse

import jwt  # PyJWT, to check the client assertion MSAL signs with the certificate below
import msal
from cryptography import x509
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.x509.oid import NameOID
from msal import SerializableTokenCache

# Offline authority: the same stand-in as Example 1, defined again here so this example runs on its own.
# It serves the endpoints MSAL calls (instance discovery and OpenID configuration), answers each POST with the next
# scripted reply, and records every request.
class FakeAuthorityResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.text = json.dumps(payload)
        self.headers = {"Content-Type": "application/json"}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")


class FakeAuthority:
    """Answers MSAL the way login.microsoftonline.com does, from a script of replies, and records every request."""
    HOST = "https://login.microsoftonline.com"

    def __init__(self, *replies):
        self.requests, self.replies = [], list(replies)

    def get(self, url, params=None, headers=None, **kwargs):
        self.requests.append(("GET", url, params, None))
        path = urllib.parse.urlsplit(url).path
        if path == "/common/discovery/instance":
            return FakeAuthorityResponse(200, {"api-version": "1.1", "metadata": [{
                "preferred_network": "login.microsoftonline.com", "preferred_cache": "login.windows.net",
                "aliases": ["login.microsoftonline.com", "login.windows.net", "login.microsoft.com", "sts.windows.net"]}]})
        assert path.endswith("/v2.0/.well-known/openid-configuration"), f"MSAL asked for an unexpected GET {url}"
        tenant = path.strip("/").split("/")[0]
        endpoints = f"{self.HOST}/{tenant}/oauth2/v2.0"
        return FakeAuthorityResponse(200, {
            "issuer": f"{self.HOST}/{tenant}/v2.0", "authorization_endpoint": f"{endpoints}/authorize",
            "token_endpoint": f"{endpoints}/token", "device_authorization_endpoint": f"{endpoints}/devicecode"})

    def post(self, url, params=None, data=None, headers=None, **kwargs):
        self.requests.append(("POST", url, params, data))
        assert self.replies, f"MSAL made a POST the example did not script: {url}"
        return FakeAuthorityResponse(*self.replies.pop(0))


# Example 2: ConfidentialClientApplication and token cache management
print("msal ConfidentialClientApplication and token cache management")

try:
    print("Example 2 - ConfidentialClientApplication and token cache management")
    
    # ConfidentialClientApplication for server applications
    print("ConfidentialClientApplication setup:")
    
    # Configuration for confidential client (web app, API, daemon)
    client_id = "confidential-client-id"
    client_secret = "client-secret-would-be-here"  # In production: use Azure Key Vault
    # A NAMED tenant: MSAL deprecates `/common` and `/organizations` for the client-credentials flow ("unreliable"), so a
    # daemon names the tenant it runs in. The fake authority serves it offline.
    tenant_id = "contoso.onmicrosoft.com"
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    token_endpoint = f"{authority}/oauth2/v2.0/token"
    
    # The app's tokens go into a serializable cache, so the cache sections below have something real to manage.
    token_cache = SerializableTokenCache()
    fake_authority = FakeAuthority((200, {
        "token_type": "Bearer", "expires_in": 3599, "ext_expires_in": 3599, "access_token": "eyJ0eXAi.fake-app-token"}))
    
    # Create confidential client application
    confidential_app = msal.ConfidentialClientApplication(
        client_id=client_id,
        client_credential=client_secret,
        authority=authority,
        token_cache=token_cache,
        http_client=fake_authority,
    )
    
    print(f"  Confidential client created")
    print(f"  Client ID: {client_id[:8]}...")
    print(f"  Authority: {authority}")
    print(f"  Application type: {type(confidential_app).__name__}")
    
    # Client credentials flow (daemon applications)
    print("Client credentials flow (for daemon applications):")
    
    # Scopes for client credentials flow use .default
    client_scopes = ["https://graph.microsoft.com/.default"]
    
    print(f"  Client credentials scopes: {client_scopes}")
    
    # The daemon's token request: one client-credentials POST to the tenant's token endpoint, carrying the app's identity
    client_result = confidential_app.acquire_token_for_client(scopes=client_scopes)
    assert client_result["access_token"] == "eyJ0eXAi.fake-app-token", client_result
    assert client_result["token_type"] == "Bearer" and client_result["token_source"] == "identity_provider", client_result
    assert "refresh_token" not in client_result, client_result  # the client-credentials flow issues no refresh token
    token_posts = [request for request in fake_authority.requests if request[0] == "POST"]
    assert [request[1] for request in token_posts] == [token_endpoint], fake_authority.requests
    assert token_posts[0][3]["grant_type"] == "client_credentials", token_posts
    assert (token_posts[0][3]["client_id"], token_posts[0][3]["client_secret"]) == (client_id, client_secret), token_posts
    assert token_posts[0][3]["scope"] == "https://graph.microsoft.com/.default", token_posts
    # ... and the next call is served from MSAL's cache: no second POST.
    cached_result = confidential_app.acquire_token_for_client(scopes=client_scopes)
    assert cached_result["access_token"] == client_result["access_token"], cached_result
    assert cached_result["token_source"] == "cache", cached_result
    assert sum(1 for request in fake_authority.requests if request[0] == "POST") == 1, fake_authority.requests
    
    print(f"  Client credentials response:")
    for key, value in client_result.items():
        if "token" in key and len(str(value)) > 20:
            print(f"    {key}: {str(value)[:25]}...")
        else:
            print(f"    {key}: {value}")
    
    # Token cache management
    print("Token cache management:")
    
    print(f"  Token cache created: {type(token_cache).__name__}")
    print(f"  Cache has changed: {token_cache.has_state_changed}")
    assert token_cache.has_state_changed  # the token acquired above was written to it
    
    # Create app with custom cache: a public client sharing the same cache sees no signed-in user, because an app-only
    # token belongs to no account
    app_with_cache = msal.PublicClientApplication(
        client_id="sample-client-id",
        authority="https://login.microsoftonline.com/common",
        token_cache=token_cache,
        http_client=FakeAuthority(),
    )
    assert app_with_cache.get_accounts() == []
    
    print(f"  Application with custom cache created")
    
    # Cache serialization (for persistent storage): serialize() is the point MSAL treats as "saved"
    cache_state = token_cache.serialize()
    assert not token_cache.has_state_changed
    assert [entry["secret"] for entry in json.loads(cache_state)["AccessToken"].values()] == ["eyJ0eXAi.fake-app-token"]
    print(f"  Serialized cache size: {len(cache_state)} characters")
    
    # Cache deserialization: a restarted daemon loads the saved state and is served from it, with no token request
    new_cache = SerializableTokenCache()
    new_cache.deserialize(cache_state)
    restarted_app = msal.ConfidentialClientApplication(
        client_id=client_id, client_credential=client_secret, authority=authority,
        token_cache=new_cache, http_client=FakeAuthority())
    restored_result = restarted_app.acquire_token_for_client(scopes=client_scopes)
    assert restored_result["access_token"] == client_result["access_token"], restored_result
    assert restored_result["token_source"] == "cache", restored_result
    print(f"  Cache deserialized successfully")
    
    # Cache change tracking. SerializableTokenCache has no callback API: MSAL's documented pattern is to check
    # `has_state_changed` after each token call and persist the cache when it is set.
    print("Cache change tracking:")
    
    saved_states = []
    
    def persist_if_changed(cache):
        """Save the cache only when MSAL reports it changed since the last save."""
        if cache.has_state_changed:
            saved_states.append(cache.serialize())
    
    persist_if_changed(new_cache)
    assert saved_states == []  # served from the cache: nothing changed, nothing to save
    
    print(f"  Cache is persisted only when has_state_changed is set")
    
    # On-behalf-of (OBO) flow
    print("On-behalf-of (OBO) flow:")
    
    # OBO flow is used when a web API needs to call another API on behalf of a user
    user_assertion = "jwt-token-from-calling-application"  # Would be actual JWT
    obo_scopes = ["https://graph.microsoft.com/User.Read"]
    
    print(f"  OBO user assertion: {user_assertion[:20]}...")
    print(f"  OBO scopes: {obo_scopes}")
    
    fake_authority.replies.append((200, {
        "token_type": "Bearer", "scope": "https://graph.microsoft.com/User.Read", "expires_in": 3599,
        "access_token": "eyJ0eXAi.fake-obo-token", "refresh_token": "0.fake-obo-refresh-token"}))
    obo_result = confidential_app.acquire_token_on_behalf_of(
        user_assertion=user_assertion,
        scopes=obo_scopes
    )
    assert obo_result["access_token"] == "eyJ0eXAi.fake-obo-token", obo_result
    assert obo_result["token_source"] == "identity_provider", obo_result
    # The request MSAL formed: the JWT-bearer grant, asking for a token to be used on behalf of the caller's user
    obo_form = [request for request in fake_authority.requests if request[0] == "POST"][-1][3]
    assert obo_form["grant_type"] == "urn:ietf:params:oauth:grant-type:jwt-bearer", obo_form
    assert obo_form["requested_token_use"] == "on_behalf_of" and obo_form["assertion"] == user_assertion, obo_form
    # The OBO tokens went into the same cache, so the persistence pattern above now saves it
    persist_if_changed(token_cache)
    assert len(saved_states) == 1 and "0.fake-obo-refresh-token" in saved_states[0], saved_states
    
    print(f"  OBO response:")
    for key, value in obo_result.items():
        if "token" in key and isinstance(value, str) and len(value) > 20:
            print(f"    {key}: {str(value)[:25]}...")
        else:
            print(f"    {key}: {value}")
    
    # Certificate-based authentication
    print("Certificate-based authentication:")
    
    # Instead of client_secret, use a certificate for higher security: MSAL signs a short-lived client assertion with
    # the private key and never sends a secret. A self-signed certificate made here stands in for the one registered
    # on the app; passing `public_certificate` makes MSAL identify it by its SHA-256 thumbprint (the SHA-1 `thumbprint`
    # form is deprecated).
    cert_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    cert_name = x509.Name([x509.NameAttribute(NameOID.COMMON_NAME, client_id)])
    cert_now = datetime.datetime.now(datetime.timezone.utc)
    certificate = (x509.CertificateBuilder().subject_name(cert_name).issuer_name(cert_name)
                   .public_key(cert_key.public_key()).serial_number(x509.random_serial_number())
                   .not_valid_before(cert_now).not_valid_after(cert_now + datetime.timedelta(days=1))
                   .sign(cert_key, hashes.SHA256()))
    cert_config = {
        "private_key": cert_key.private_bytes(serialization.Encoding.PEM, serialization.PrivateFormat.PKCS8,
                                              serialization.NoEncryption()).decode(),  # PEM format
        "public_certificate": certificate.public_bytes(serialization.Encoding.PEM).decode(),  # Public cert
    }
    
    cert_authority = FakeAuthority((200, {"token_type": "Bearer", "expires_in": 3599, "access_token": "eyJ0eXAi.fake-cert-token"}))
    cert_app = msal.ConfidentialClientApplication(
        client_id=client_id,
        client_credential=cert_config,
        authority=authority,
        http_client=cert_authority,
    )
    cert_result = cert_app.acquire_token_for_client(scopes=client_scopes)
    assert cert_result["access_token"] == "eyJ0eXAi.fake-cert-token", cert_result
    cert_form = [request for request in cert_authority.requests if request[0] == "POST"][0][3]
    assert "client_secret" not in cert_form, cert_form
    assert cert_form["client_assertion_type"] == "urn:ietf:params:oauth:client-assertion-type:jwt-bearer", cert_form
    # The assertion verifies against the certificate's key, is addressed to the token endpoint, names the app, and its
    # header carries the certificate's SHA-256 thumbprint
    assertion_header = jwt.get_unverified_header(cert_form["client_assertion"])
    assertion_claims = jwt.decode(cert_form["client_assertion"], cert_key.public_key(), algorithms=["PS256"],
                                  audience=token_endpoint)
    assert assertion_header["alg"] == "PS256", assertion_header
    assert base64.urlsafe_b64decode(assertion_header["x5t#S256"]) == certificate.fingerprint(hashes.SHA256()), assertion_header
    assert assertion_claims["iss"] == assertion_claims["sub"] == client_id, assertion_claims
    
    print(f"  Certificate SHA-256 thumbprint: {certificate.fingerprint(hashes.SHA256()).hex()[:16]}...")
    print(f"  Using certificate instead of client secret for authentication")
    
    # Application types and use cases
    print("Application types and use cases:")
    
    app_types = {
        "Desktop/Mobile Apps": {
            "type": "PublicClientApplication",
            "flows": ["Interactive", "Device Code", "Integrated Windows Auth"],
            "use_cases": ["Desktop apps", "Mobile apps", "CLI tools"]
        },
        "Web Apps": {
            "type": "ConfidentialClientApplication",
            "flows": ["Authorization Code", "On-behalf-of"],
            "use_cases": ["Web applications", "Web APIs", "Server-side apps"]
        },
        "Daemon/Service Apps": {
            "type": "ConfidentialClientApplication",
            "flows": ["Client Credentials"],
            "use_cases": ["Background services", "Scheduled tasks", "API-to-API"]
        }
    }
    
    for app_type, details in app_types.items():
        print(f"  {app_type}:")
        print(f"    MSAL Type: {details['type']}")
        print(f"    Auth Flows: {', '.join(details['flows'])}")
        print(f"    Use Cases: {', '.join(details['use_cases'])}")
    
    print("msal example 2 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    # Every call above talks to an in-cell FakeAuthority, so this example needs no network: any failure is a defect.
    raise AssertionError(
        f"msal example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Device code flow and error handling patterns

In [ ]:
import base64
import json
import time
import urllib.parse

import msal

class FakeAuthorityResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.text = json.dumps(payload)
        self.headers = {"Content-Type": "application/json"}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")

class FakeAuthority:
    """Answers MSAL the way login.microsoftonline.com does, from a script of replies, and records every request."""
    HOST = "https://login.microsoftonline.com"

    def __init__(self, *replies):
        self.requests, self.replies = [], list(replies)

    def get(self, url, params=None, headers=None, **kwargs):
        self.requests.append(("GET", url, params, None))
        path = urllib.parse.urlsplit(url).path
        if path == "/common/discovery/instance":
            return FakeAuthorityResponse(200, {"api-version": "1.1", "metadata": [{
                "preferred_network": "login.microsoftonline.com", "preferred_cache": "login.windows.net",
                "aliases": ["login.microsoftonline.com", "login.windows.net", "login.microsoft.com", "sts.windows.net"]}]})
        assert path.endswith("/v2.0/.well-known/openid-configuration"), f"MSAL asked for an unexpected GET {url}"
        tenant = path.strip("/").split("/")[0]
        endpoints = f"{self.HOST}/{tenant}/oauth2/v2.0"
        return FakeAuthorityResponse(200, {
            "issuer": f"{self.HOST}/{tenant}/v2.0", "authorization_endpoint": f"{endpoints}/authorize",
            "token_endpoint": f"{endpoints}/token", "device_authorization_endpoint": f"{endpoints}/devicecode"})

    def post(self, url, params=None, data=None, headers=None, **kwargs):
        self.requests.append(("POST", url, params, data))
        assert self.replies, f"MSAL made a POST the example did not script: {url}"
        return FakeAuthorityResponse(*self.replies.pop(0))

def b64url_json(payload):
    return base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip("=")

# Example 3: Device code flow and error handling patterns
print("msal device code flow and error handling patterns")

try:
    print("Example 3 - Device code flow and error handling patterns")
    
    # Device code flow for browserless environments
    print("Device code flow implementation:")
    
    # What the authority answers: a user code from the device endpoint (with a 1-second poll interval, so the example
    # stays quick — Microsoft's is 5), then `authorization_pending` twice while the user signs in elsewhere, then tokens.
    device_code_reply = {
        "device_code": "BAQABAAEAAADXzZ3ifr-fake",
        "user_code": "ABCD1234",
        "verification_uri": "https://microsoft.com/devicelogin",
        "expires_in": 900,
        "interval": 1,
        "message": "To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code ABCD1234 to authenticate."
    }
    pending_reply = (400, {"error": "authorization_pending", "error_codes": [70016],
                           "error_description": "AADSTS70016: OAuth 2.0 device flow error. Authorization is pending. Continue polling."})
    fake_authority = FakeAuthority((200, device_code_reply), pending_reply, pending_reply, (200, {
        "token_type": "Bearer", "scope": "User.Read Files.Read openid profile", "expires_in": 3599,
        "access_token": "eyJ0eXAi.fake-device-token", "refresh_token": "0.fake-device-refresh-token",
        "client_info": b64url_json({"uid": "00000000-0000-0000-0000-00000000abcd", "utid": "72f988bf-86f1-41af-91ab-2d7cd011db47"}),
        "id_token": b64url_json({"alg": "none", "typ": "JWT"}) + "." + b64url_json({
            "aud": "device-flow-client-id", "iss": "https://login.microsoftonline.com/72f988bf-86f1-41af-91ab-2d7cd011db47/v2.0",
            "tid": "72f988bf-86f1-41af-91ab-2d7cd011db47", "oid": "00000000-0000-0000-0000-00000000abcd",
            "preferred_username": "user1@example.com"}) + "."}))
    
    # Create public client for device code flow
    device_app = msal.PublicClientApplication(
        client_id="device-flow-client-id",
        authority="https://login.microsoftonline.com/common",
        http_client=fake_authority,
    )
    
    print(f"  Device flow app created: {type(device_app).__name__}")
    
    # Scopes for device flow
    device_scopes = ["User.Read", "Files.Read"]
    
    print(f"  Requested scopes: {device_scopes}")
    
    # Initiate device flow: MSAL asks the device endpoint for a code the user enters on another device
    flow = device_app.initiate_device_flow(scopes=device_scopes)
    assert flow["user_code"] == "ABCD1234" and flow["verification_uri"] == "https://microsoft.com/devicelogin", flow
    assert flow["interval"] == 1 and flow["expires_at"] > time.time(), flow
    device_request = fake_authority.requests[-1]
    assert device_request[:2] == ("POST", "https://login.microsoftonline.com/common/oauth2/v2.0/devicecode"), device_request
    assert set(device_request[3]["scope"].split()) == set(device_scopes) | {"offline_access", "openid", "profile"}, device_request
    
    print(f"  Device flow initiated:")
    for key in ("user_code", "verification_uri", "expires_in", "interval", "message"):
        print(f"    {key}: {flow[key]}")
    
    # Device flow polling: MSAL polls the token endpoint every `interval` seconds while the authority answers
    # `authorization_pending`, and returns once the user has signed in
    print("Device flow polling process:")
    
    token_result = device_app.acquire_token_by_device_flow(flow)
    token_polls = [request for request in fake_authority.requests if request[1].endswith("/oauth2/v2.0/token")]
    assert len(token_polls) == 3, fake_authority.requests  # two pending answers, then the tokens
    for poll in token_polls:
        assert poll[3]["grant_type"] == "urn:ietf:params:oauth:grant-type:device_code", poll
        assert poll[3]["device_code"] == device_code_reply["device_code"], poll
    assert token_result["access_token"] == "eyJ0eXAi.fake-device-token", token_result
    assert token_result["token_source"] == "identity_provider", token_result
    print(f"    Polled the token endpoint {len(token_polls)} times")
    
    print(f"  Device flow completed successfully")
    print(f"  Token expires in: {token_result['expires_in']} seconds")
    
    # Error handling patterns. MSAL does not RAISE when the authority refuses: it returns a result dict carrying
    # `error`, `error_description` and `error_codes`. One poll each (`exit_condition=lambda flow: True` stops MSAL's
    # loop after the first answer), against a fresh app and fake authority, shows the documented shape per refusal.
    print("MSAL error handling patterns:")
    
    refusal_replies = {
        "expired_token": (400, {"error": "expired_token", "error_codes": [70019],
                                "error_description": "AADSTS70019: Verification code expired."}),
        "invalid_client": (401, {"error": "invalid_client", "error_codes": [7000218],
                                 "error_description": "AADSTS7000218: The request body must contain client_assertion or client_secret."}),
        "consent_required": (400, {"error": "invalid_grant", "suberror": "consent_required", "error_codes": [65001],
                                   "error_description": "AADSTS65001: The user or administrator has not consented to use the application."}),
        "authorization_pending": pending_reply,
    }
    refusals = {}
    for scenario, reply in refusal_replies.items():
        refusing_app = msal.PublicClientApplication(
            "device-flow-client-id", authority="https://login.microsoftonline.com/common",
            http_client=FakeAuthority((200, device_code_reply), reply))
        refusal = refusing_app.acquire_token_by_device_flow(
            refusing_app.initiate_device_flow(scopes=device_scopes), exit_condition=lambda flow: True)
        assert "access_token" not in refusal, refusal
        assert refusal["error"] == reply[1]["error"] and refusal["error_codes"] == reply[1]["error_codes"], refusal
        assert refusal.get("suberror") == reply[1].get("suberror"), refusal
        refusals[scenario] = refusal
    
    print(f"  Refusals as MSAL returns them:")
    for scenario, refusal in refusals.items():
        print(f"    {scenario}: {refusal['error']} {refusal['error_codes']}")
    
    # Error handling implementation
    print("Error handling implementation:")
    
    def handle_msal_error(result):
        """Handle MSAL authentication errors"""
        if "error" not in result:
            return {"status": "success", "tokens": result}
        
        # Consent and similar conditions arrive as `invalid_grant` with a `suberror` naming the case
        error = result.get("suberror") or result.get("error")
        error_description = result.get("error_description", "No description")
        
        error_responses = {
            "invalid_client": {
                "action": "check_configuration",
                "message": "Verify client ID and app registration",
                "retry": False
            },
            "consent_required": {
                "action": "request_consent",
                "message": "User or admin consent needed",
                "retry": True,
                "consent_url": result.get("consent_url")
            },
            "authorization_pending": {
                "action": "continue_polling",
                "message": "Continue device flow polling",
                "retry": True,
                "wait_seconds": result.get("interval", 5)
            },
            "expired_token": {
                "action": "restart_device_flow",
                "message": "The user code expired; start a new device flow",
                "retry": True
            }
        }
        
        response = error_responses.get(error, {
            "action": "unknown_error",
            "message": error_description,
            "retry": False
        })
        
        return {
            "status": "error",
            "error_type": error,
            "error_description": error_description,
            "recommended_action": response["action"],
            "message": response["message"],
            "can_retry": response["retry"]
        }
    
    # Test error handling on the results MSAL actually returned above
    expected_actions = {
        "expired_token": ("restart_device_flow", True),
        "invalid_client": ("check_configuration", False),
        "consent_required": ("request_consent", True),
        "authorization_pending": ("continue_polling", True),
    }
    
    print(f"  Error handling test results:")
    for scenario, refusal in refusals.items():
        handled = handle_msal_error(refusal)
        assert handled["status"] == "error", handled
        assert (handled["recommended_action"], handled["can_retry"]) == expected_actions[scenario], (scenario, handled)
        print(f"    {scenario}: {handled['status']}")
        print(f"      Action: {handled['recommended_action']}")
        print(f"      Message: {handled['message']}")
        print(f"      Can retry: {handled['can_retry']}")
    assert handle_msal_error(token_result)["status"] == "success"
    
    # Silent authentication patterns
    print("Silent authentication patterns:")
    
    def attempt_silent_authentication(app, accounts, scopes):
        """Attempt silent token acquisition"""
        if not accounts:
            return {"status": "no_accounts", "message": "No cached accounts available"}
        
        # Try silent acquisition for each account
        for account in accounts:
            result = app.acquire_token_silent(scopes=scopes, account=account)
            
            if result and result.get("access_token"):
                return {
                    "status": "success",
                    "account": account.get("username", "unknown"),
                    "token": result
                }
        
        return {"status": "silent_failed", "message": "Silent acquisition failed for all accounts"}
    
    # The account the device flow signed in is in the app's cache; a silent call is served from it with no token request
    signed_in_accounts = device_app.get_accounts()
    assert [account["username"] for account in signed_in_accounts] == ["user1@example.com"], signed_in_accounts
    posts_before_silent = sum(1 for request in fake_authority.requests if request[0] == "POST")
    
    silent_result = attempt_silent_authentication(device_app, signed_in_accounts, device_scopes)
    assert silent_result["status"] == "success" and silent_result["account"] == "user1@example.com", silent_result
    assert silent_result["token"]["token_source"] == "cache", silent_result
    assert sum(1 for request in fake_authority.requests if request[0] == "POST") == posts_before_silent
    assert attempt_silent_authentication(device_app, [], device_scopes)["status"] == "no_accounts"
    print(f"  Silent authentication result: {silent_result['status']}")
    if silent_result['status'] == 'success':
        print(f"    Account: {silent_result['account']}")
        print(f"    Token expires in: {silent_result['token']['expires_in']} seconds")
    
    # Logging and telemetry
    print("Logging and telemetry:")
    
    # MSAL provides built-in logging
    import logging
    
    # Configure MSAL logging (would be done at application startup)
    msal_logger = logging.getLogger("msal")
    msal_logger.setLevel(logging.INFO)
    
    print(f"  MSAL logging configured at INFO level")
    print(f"  Logger name: {msal_logger.name}")
    print(f"  Logger level: {msal_logger.level}")
    
    # Custom telemetry for monitoring
    telemetry_events = [
        {"event": "authentication_started", "method": "device_flow", "timestamp": time.time()},
        {"event": "token_acquired", "method": "silent", "timestamp": time.time()},
        {"event": "error_occurred", "error": "consent_required", "timestamp": time.time()}
    ]
    
    print(f"  Sample telemetry events:")
    for event in telemetry_events:
        print(f"    {event['event']}: {event.get('method', event.get('error', 'N/A'))}")
    
    print("msal example 3 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Token validation and Microsoft Graph integration

In [ ]:
import base64
import json
import time
import urllib.parse

import jwt  # PyJWT: signs the access token below and validates it
import msal
from cryptography.hazmat.primitives.asymmetric import rsa

class FakeAuthorityResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.text = json.dumps(payload)
        self.headers = {"Content-Type": "application/json"}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")

class FakeAuthority:
    """Answers MSAL the way login.microsoftonline.com does, from a script of replies, and records every request."""
    HOST = "https://login.microsoftonline.com"

    def __init__(self, *replies):
        self.requests, self.replies = [], list(replies)

    def get(self, url, params=None, headers=None, **kwargs):
        self.requests.append(("GET", url, params, None))
        path = urllib.parse.urlsplit(url).path
        if path == "/common/discovery/instance":
            return FakeAuthorityResponse(200, {"api-version": "1.1", "metadata": [{
                "preferred_network": "login.microsoftonline.com", "preferred_cache": "login.windows.net",
                "aliases": ["login.microsoftonline.com", "login.windows.net", "login.microsoft.com", "sts.windows.net"]}]})
        assert path.endswith("/v2.0/.well-known/openid-configuration"), f"MSAL asked for an unexpected GET {url}"
        tenant = path.strip("/").split("/")[0]
        endpoints = f"{self.HOST}/{tenant}/oauth2/v2.0"
        return FakeAuthorityResponse(200, {
            "issuer": f"{self.HOST}/{tenant}/v2.0", "authorization_endpoint": f"{endpoints}/authorize",
            "token_endpoint": f"{endpoints}/token", "device_authorization_endpoint": f"{endpoints}/devicecode"})

    def post(self, url, params=None, data=None, headers=None, **kwargs):
        self.requests.append(("POST", url, params, data))
        assert self.replies, f"MSAL made a POST the example did not script: {url}"
        return FakeAuthorityResponse(*self.replies.pop(0))

def b64url_json(payload):
    return base64.urlsafe_b64encode(json.dumps(payload).encode()).decode().rstrip("=")

# Example 4: Token validation and Microsoft Graph integration
print("msal token validation and Microsoft Graph integration")

try:
    print("Example 4 - Token validation and Microsoft Graph integration")
    
    # Token validation and inspection
    print("JWT token validation and inspection:")
    
    # A stand-in for the tenant's token-signing key: an RSA key pair made here. Microsoft publishes the real public keys
    # at the tenant's `jwks_uri`; the validation below is the same PyJWT call either way.
    signing_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    now = int(time.time())
    
    # JWT access token structure: the header and the claims of the token signed below
    sample_jwt_header = {
        "alg": "RS256",
        "kid": "key-identifier", 
        "typ": "JWT"
    }
    
    sample_jwt_payload = {
        "aud": "https://graph.microsoft.com",
        "iss": "https://sts.windows.net/tenant-id/",
        "iat": now,
        "exp": now + 3600,
        "sub": "user-subject-id",
        "upn": "user@example.com",
        "name": "John Doe",
        "tid": "tenant-id",
        "scp": "User.Read Mail.Read Calendars.Read",
        "roles": ["User.ReadBasic.All"],
        "ver": "2.0"
    }
    access_token = jwt.encode(sample_jwt_payload, signing_key, algorithm="RS256", headers={"kid": "key-identifier"})
    assert jwt.get_unverified_header(access_token) == sample_jwt_header
    
    print(f"  JWT Header: {sample_jwt_header}")
    print(f"  JWT Payload (selected fields):")
    important_fields = ["aud", "iss", "exp", "upn", "scp", "roles"]
    for field in important_fields:
        if field in sample_jwt_payload:
            print(f"    {field}: {sample_jwt_payload[field]}")
    
    # Token validation function
    def validate_access_token(token, expected_audience=None):
        """Verify the token's signature, expiry, audience and issuer; report the verdict"""
        try:
            claims = jwt.decode(token, signing_key.public_key(), algorithms=["RS256"],
                                audience=expected_audience, issuer="https://sts.windows.net/tenant-id/")
        except jwt.InvalidTokenError as rejection:
            return {"valid": False, "checks": [f"Rejected: {type(rejection).__name__}"], "claims": None}
        return {"valid": True, "checks": ["Signature valid", "Token not expired", "Audience valid", "Issuer valid"],
                "claims": claims}
    
    # Test token validation
    validation = validate_access_token(
        access_token, 
        expected_audience="https://graph.microsoft.com"
    )
    assert validation["valid"] and validation["claims"] == sample_jwt_payload, validation
    
    print(f"  Token validation result: {'✓ VALID' if validation['valid'] else '✗ INVALID'}")
    for check in validation['checks']:
        print(f"    • {check}")
    
    # ... and what it must refuse: an expired token, a token for another API, and one signed by a different key
    other_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
    refusal_cases = {
        "expired": (jwt.encode(dict(sample_jwt_payload, exp=now - 60), signing_key, algorithm="RS256"),
                    "https://graph.microsoft.com", "ExpiredSignatureError"),
        "wrong audience": (access_token, "api://my-api", "InvalidAudienceError"),
        "foreign signature": (jwt.encode(sample_jwt_payload, other_key, algorithm="RS256"),
                              "https://graph.microsoft.com", "InvalidSignatureError"),
    }
    for label, (token, audience, expected_rejection) in refusal_cases.items():
        verdict = validate_access_token(token, expected_audience=audience)
        assert not verdict["valid"] and verdict["checks"] == [f"Rejected: {expected_rejection}"], (label, verdict)
        print(f"    {label}: rejected ({expected_rejection})")
    
    # Microsoft Graph API integration patterns
    print("Microsoft Graph API integration:")
    
    # Graph API endpoints and methods
    graph_endpoints = {
        "user_profile": {
            "method": "GET",
            "url": "https://graph.microsoft.com/v1.0/me",
            "scope_required": "User.Read",
            "description": "Get current user's profile"
        },
        "user_mail": {
            "method": "GET", 
            "url": "https://graph.microsoft.com/v1.0/me/messages",
            "scope_required": "Mail.Read",
            "description": "Get user's mail messages"
        },
        "user_calendar": {
            "method": "GET",
            "url": "https://graph.microsoft.com/v1.0/me/events",
            "scope_required": "Calendars.Read",
            "description": "Get user's calendar events"
        },
        "user_files": {
            "method": "GET",
            "url": "https://graph.microsoft.com/v1.0/me/drive/root/children",
            "scope_required": "Files.Read",
            "description": "Get user's OneDrive files"
        },
        "send_mail": {
            "method": "POST",
            "url": "https://graph.microsoft.com/v1.0/me/sendMail",
            "scope_required": "Mail.Send",
            "description": "Send email on behalf of user"
        }
    }
    
    print(f"  Common Graph API endpoints:")
    for endpoint_name, details in graph_endpoints.items():
        print(f"    {endpoint_name}:")
        print(f"      Method: {details['method']}")
        print(f"      URL: {details['url']}")
        print(f"      Required scope: {details['scope_required']}")
        print(f"      Description: {details['description']}")
    
    # Graph API client implementation
    class GraphAPIClient:
        """Microsoft Graph API client with MSAL integration"""
        
        def __init__(self, msal_app, account):
            self.msal_app = msal_app
            self.account = account
            self.graph_base_url = "https://graph.microsoft.com/v1.0"
            self.last_request = None
        
        def get_access_token(self, scopes):
            """Get access token for Graph API calls from MSAL's cache"""
            result = self.msal_app.acquire_token_silent(scopes, account=self.account)
            if result and "access_token" in result:
                return result["access_token"]
            
            # Nothing cached for these scopes: an interactive app would now call
            # self.msal_app.acquire_token_interactive(scopes), which opens a browser for the user
            raise LookupError(f"no cached token for {scopes}; the user must sign in again for them")
        
        def make_graph_request(self, endpoint, method="GET", data=None, scopes=None):
            """Make authenticated request to Microsoft Graph"""
            if not scopes:
                scopes = ["User.Read"]  # Default scope
            
            # Get access token
            access_token = self.get_access_token(scopes)
            
            # Prepare request headers
            headers = {
                "Authorization": f"Bearer {access_token}",
                "Content-Type": "application/json",
                "Accept": "application/json"
            }
            
            # Full URL
            url = f"{self.graph_base_url}{endpoint}"
            
            # Simulate request (in real implementation, use requests library)
            request_info = {
                "method": method,
                "url": url,
                "headers": headers,
                "data": data
            }
            
            self.last_request = request_info
            
            return self.simulate_graph_response(endpoint, method)
        
        def simulate_graph_response(self, endpoint, method):
            """Simulate Graph API responses"""
            responses = {
                "/me": {
                    "id": "12345678-1234-1234-1234-123456789012",
                    "displayName": "John Doe",
                    "mail": "john@example.com",
                    "userPrincipalName": "john@example.com",
                    "jobTitle": "Software Developer",
                    "officeLocation": "Building 1"
                },
                "/me/messages": {
                    "value": [
                        {
                            "id": "msg1",
                            "subject": "Meeting Reminder",
                            "from": {"emailAddress": {"address": "colleague@example.com"}},
                            "receivedDateTime": "2024-01-01T10:00:00Z"
                        },
                        {
                            "id": "msg2", 
                            "subject": "Project Update",
                            "from": {"emailAddress": {"address": "manager@example.com"}},
                            "receivedDateTime": "2024-01-01T14:30:00Z"
                        }
                    ]
                },
                "/me/events": {
                    "value": [
                        {
                            "id": "event1",
                            "subject": "Team Meeting",
                            "start": {"dateTime": "2024-01-02T09:00:00Z"},
                            "end": {"dateTime": "2024-01-02T10:00:00Z"},
                            "location": {"displayName": "Conference Room A"}
                        }
                    ]
                }
            }
            
            return responses.get(endpoint, {"message": "Endpoint not found in simulation"})
    
    # Test Graph API client. The user signs in once with the device flow (the fake authority grants at the first
    # poll), so MSAL's cache holds the RS256 token signed above, and the client serves every call from that cache.
    fake_authority = FakeAuthority(
        (200, {"device_code": "graph-device-code", "user_code": "GRAPH123", "expires_in": 900, "interval": 1,
               "verification_uri": "https://microsoft.com/devicelogin",
               "message": "To sign in, open https://microsoft.com/devicelogin and enter the code GRAPH123."}),
        (200, {"token_type": "Bearer", "scope": "User.Read Mail.Read Calendars.Read", "expires_in": 3599,
               "access_token": access_token, "client_info": b64url_json({"uid": "user-subject-id", "utid": "tenant-id"}),
               "id_token": b64url_json({"alg": "none", "typ": "JWT"}) + "." + b64url_json({
                   "aud": "sample-client-id", "iss": "https://login.microsoftonline.com/tenant-id/v2.0",
                   "tid": "tenant-id", "oid": "user-subject-id", "preferred_username": "user@example.com"}) + "."}))
    sample_app = msal.PublicClientApplication("sample-client-id", http_client=fake_authority)
    sign_in = sample_app.acquire_token_by_device_flow(
        sample_app.initiate_device_flow(scopes=["User.Read", "Mail.Read", "Calendars.Read"]))
    assert sign_in["access_token"] == access_token and sign_in["token_source"] == "identity_provider", sign_in
    graph_client = GraphAPIClient(sample_app, sample_app.get_accounts()[0])
    
    # Test different Graph API calls
    print(f"  Graph API client testing:")
    
    # Get user profile
    user_profile = graph_client.make_graph_request("/me", scopes=["User.Read"])
    print(f"    User profile: {user_profile.get('displayName')} ({user_profile.get('mail')})")
    
    # Get user messages
    messages = graph_client.make_graph_request("/me/messages", scopes=["Mail.Read"])
    print(f"    Messages: {len(messages.get('value', []))} found")
    for msg in messages.get('value', [])[:2]:
        print(f"      • {msg['subject']} from {msg['from']['emailAddress']['address']}")
    
    # Get user events
    events = graph_client.make_graph_request("/me/events", scopes=["Calendars.Read"])
    print(f"    Events: {len(events.get('value', []))} found")
    for event in events.get('value', [])[:2]:
        print(f"      • {event['subject']} at {event['start']['dateTime']}")
    
    # Every call carried the RS256 token MSAL served from its cache: after the sign-in (one device-code POST and one
    # token POST) nothing went back to the authority
    assert graph_client.last_request["headers"]["Authorization"] == f"Bearer {access_token}", graph_client.last_request
    assert graph_client.last_request["url"] == "https://graph.microsoft.com/v1.0/me/events", graph_client.last_request
    assert sum(1 for request in fake_authority.requests if request[0] == "POST") == 2, fake_authority.requests
    # A scope the user never granted has no cached token, and the client says so instead of inventing one
    try:
        graph_client.make_graph_request("/me/drive/root/children", scopes=["Files.Read"])
    except LookupError as missing:
        assert "Files.Read" in str(missing), missing
    else:
        raise AssertionError("the Graph client produced a token for a scope that was never granted")
    print(f"    Files.Read: no cached token, so the client asks the user to sign in again")
    
    # Batch requests for efficiency
    print("Batch requests for efficiency:")
    
    def create_batch_request(requests):
        """Create batch request for multiple Graph API calls"""
        batch_request = {
            "requests": []
        }
        
        for i, request in enumerate(requests):
            batch_item = {
                "id": str(i + 1),
                "method": request.get("method", "GET"),
                "url": request["url"],
                "headers": request.get("headers", {})
            }
            batch_request["requests"].append(batch_item)
        
        return batch_request
    
    # Sample batch requests
    batch_requests = [
        {"url": "/me", "method": "GET"},
        {"url": "/me/messages?$top=5", "method": "GET"},
        {"url": "/me/events?$top=5", "method": "GET"}
    ]
    
    batch_payload = create_batch_request(batch_requests)
    print(f"    Batch request with {len(batch_payload['requests'])} operations")
    for request in batch_payload['requests']:
        print(f"      {request['id']}: {request['method']} {request['url']}")
    
    # Permission and scope management
    print("Permission and scope management:")
    
    scope_categories = {
        "User permissions": [
            "User.Read - Read user profile",
            "User.ReadWrite - Read and write user profile",
            "User.ReadBasic.All - Read basic profiles of all users"
        ],
        "Mail permissions": [
            "Mail.Read - Read user mail", 
            "Mail.ReadWrite - Read and write user mail",
            "Mail.Send - Send mail as user"
        ],
        "Calendar permissions": [
            "Calendars.Read - Read user calendars",
            "Calendars.ReadWrite - Read and write user calendars"
        ],
        "Files permissions": [
            "Files.Read - Read user files",
            "Files.ReadWrite - Read and write user files",
            "Files.Read.All - Read all files user can access"
        ]
    }
    
    for category, scopes in scope_categories.items():
        print(f"    {category}:")
        for scope in scopes:
            print(f"      • {scope}")
    
    # Rate limiting and throttling
    print("Rate limiting and best practices:")
    
    best_practices = [
        "Implement exponential backoff for 429 (Too Many Requests) responses",
        "Use batch requests to reduce API call count", 
        "Cache responses when appropriate to minimize redundant calls",
        "Request only necessary fields using $select parameter",
        "Use delta queries for incremental data synchronization",
        "Respect Retry-After header in throttling responses",
        "Monitor usage through Graph API usage reports",
        "Implement proper error handling for all response codes"
    ]
    
    for practice in best_practices:
        print(f"    • {practice}")
    
    print("msal example 4 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Production deployment patterns and comprehensive MSAL capabilities

In [ ]:
import msal
from msal import SerializableTokenCache
import json
import time
import threading
import urllib.parse
import os
import tempfile

class FakeAuthorityResponse:
    def __init__(self, status_code, payload):
        self.status_code = status_code
        self.text = json.dumps(payload)
        self.headers = {"Content-Type": "application/json"}

    def raise_for_status(self):
        if self.status_code >= 400:
            raise RuntimeError(f"HTTP {self.status_code}")

class FakeAuthority:
    """Answers MSAL the way login.microsoftonline.com does — including REFUSING an unregistered client secret."""
    HOST = "https://login.microsoftonline.com"

    def __init__(self):
        self.requests = []

    def get(self, url, params=None, headers=None, **kwargs):
        self.requests.append(("GET", url, params, None))
        path = urllib.parse.urlsplit(url).path
        if path == "/common/discovery/instance":
            return FakeAuthorityResponse(200, {"api-version": "1.1", "metadata": [{
                "preferred_network": "login.microsoftonline.com", "preferred_cache": "login.windows.net",
                "aliases": ["login.microsoftonline.com", "login.windows.net", "login.microsoft.com", "sts.windows.net"]}]})
        tenant = path.strip("/").split("/")[0]
        assert path == f"/{tenant}/v2.0/.well-known/openid-configuration", f"MSAL asked for an unexpected GET {url}"
        return FakeAuthorityResponse(200, {
            "issuer": f"{self.HOST}/{tenant}/v2.0",
            "authorization_endpoint": f"{self.HOST}/{tenant}/oauth2/v2.0/authorize",
            "token_endpoint": f"{self.HOST}/{tenant}/oauth2/v2.0/token",
        })

    def post(self, url, params=None, data=None, headers=None, **kwargs):
        self.requests.append(("POST", url, params, data))
        return FakeAuthorityResponse(401, {
            "error": "invalid_client", "error_codes": [7000215],
            "error_description": "AADSTS7000215: Invalid client secret provided."})

class TokenAcquisitionError(RuntimeError):
    """MSAL returned its documented failure shape — a result dict carrying `error` — instead of a token."""
    def __init__(self, result):
        super().__init__(f"{result.get('error')} - {result.get('error_description')}")
        self.result = result

# Example 5: Production deployment patterns and comprehensive MSAL capabilities
print("msal production deployment patterns and comprehensive capabilities")

try:
    print("Example 5 - Production deployment patterns and comprehensive capabilities")
    
    # Package overview
    print("MSAL Package Overview:")
    print(f"  Package type: {type(msal).__name__}")
    
    if hasattr(msal, '__version__'):
        print(f"  Version: {msal.__version__}")
    
    # Key components
    msal_components = {
        'PublicClientApplication': 'Desktop/mobile app authentication',
        'ConfidentialClientApplication': 'Web app/API/daemon authentication', 
        'SerializableTokenCache': 'Persistent token storage',
        'TokenCache': 'In-memory token storage'
    }
    
    print(f"  Key Components:")
    for component, description in msal_components.items():
        try:
            comp_class = getattr(msal, component)
            print(f"    ✓ {component}: {description}")
        except AttributeError:
            print(f"    ○ {component}: {description}")
    
    # Production deployment patterns
    print("Production Deployment Patterns:")
    
    # 1. Web application pattern
    print("  1. Web Application Pattern:")
    
    class ProductionWebApp:
        """Production web application with MSAL integration"""
        
        def __init__(self, config):
            self.config = config
            
            # Initialize token cache with persistence
            self.token_cache = SerializableTokenCache()
            
            # Load existing cache if available
            cache_file = config.get("token_cache_file", "token_cache.json")
            try:
                with open(cache_file, 'r') as cache_f:
                    cache_data = cache_f.read()
                    if cache_data:
                        self.token_cache.deserialize(cache_data)
            except FileNotFoundError:
                pass  # New cache
            
            # Initialize MSAL app
            self.msal_app = msal.ConfidentialClientApplication(
                client_id=config["client_id"],
                client_credential=config["client_secret"],
                authority=config["authority"],
                token_cache=self.token_cache,
                http_client=config["http_client"]
            )
            
            # SerializableTokenCache has no change-listener API; MSAL's documented pattern is to check
            # `has_state_changed` after each token call and persist the cache when it is set.
            self.persist_cache_if_changed()
        
        def persist_cache_if_changed(self):
            """Persist the token cache when MSAL reports it changed."""
            if self.token_cache.has_state_changed:
                # Save cache to persistent storage
                cache_file = self.config.get("token_cache_file", "token_cache.json")
                with open(cache_file, 'w') as cache_f:
                    cache_f.write(self.token_cache.serialize())
        
        def begin_sign_in(self, scopes, redirect_uri, state=None):
            """Start the authorization code flow (+ PKCE). The flow MSAL returns holds the state, nonce and PKCE verifier:
            keep it in the user's session until the authority's reply arrives. `form_post` has the authority POST the
            code back instead of putting it in the URL. (This replaces the deprecated get_authorization_request_url.)"""
            return self.msal_app.initiate_auth_code_flow(
                scopes,
                redirect_uri=redirect_uri,
                state=state,
                response_mode="form_post"
            )
        
        def handle_auth_response(self, flow, auth_response):
            """Redeem the authority's reply. MSAL checks its `state` against the flow first (a mismatch raises ValueError);
            a reply carrying `error` (the user declined) comes back as MSAL's result dict with no request made.
            (This replaces the deprecated acquire_token_by_authorization_code.)"""
            result = self.msal_app.acquire_token_by_auth_code_flow(flow, auth_response)
            self.persist_cache_if_changed()
            return result
    
    # Sample web app configuration
    web_config = {
        "client_id": "web-app-client-id",
        "client_secret": "web-app-client-secret",  # Use Azure Key Vault in production
        "authority": "https://login.microsoftonline.com/common",
        # A private directory, so the example never reads or writes a cache file in the working directory.
        "token_cache_file": os.path.join(tempfile.mkdtemp(prefix="msal-web-app-"), "web_app_cache.json"),
        "http_client": FakeAuthority()
    }
    
    web_app = ProductionWebApp(web_config)
    # Construction resolves the authority: exactly one OpenID discovery GET, for the configured tenant.
    assert [request[1] for request in web_config["http_client"].requests] == [
        "https://login.microsoftonline.com/common/v2.0/.well-known/openid-configuration"
    ], web_config["http_client"].requests
    print(f"     Web app initialized with cache persistence")
    
    # Start a sign-in: MSAL builds the URL the user visits, plus the flow the app keeps for the reply
    flow = web_app.begin_sign_in(
        scopes=["User.Read", "Mail.Read"], 
        redirect_uri="https://your-app.com/auth/callback",
        state="random-state-string"
    )
    auth_url = flow["auth_uri"]
    auth_query = urllib.parse.parse_qs(urllib.parse.urlsplit(auth_url).query)
    assert auth_url.startswith("https://login.microsoftonline.com/common/oauth2/v2.0/authorize?"), auth_url
    assert auth_query["client_id"] == ["web-app-client-id"] and auth_query["response_type"] == ["code"], auth_query
    assert auth_query["state"] == ["random-state-string"] == [flow["state"]], auth_query
    assert auth_query["redirect_uri"] == ["https://your-app.com/auth/callback"], auth_query
    assert auth_query["response_mode"] == ["form_post"] and auth_query["code_challenge_method"] == ["S256"], auth_query
    assert {"User.Read", "Mail.Read", "offline_access", "openid", "profile"} == set(auth_query["scope"][0].split()), auth_query
    print(f"     Auth URL generated: {auth_url[:50]}...")
    
    # A reply whose state is not this flow's is refused before any token request (CSRF protection), and a user who
    # declined consent comes back as MSAL's result dict — neither reaches the authority.
    web_authority = web_config["http_client"]
    requests_before_reply = len(web_authority.requests)
    try:
        web_app.handle_auth_response(dict(flow), {"code": "fake-auth-code", "state": "forged-state"})
    except ValueError as refusal:
        assert "state mismatch" in str(refusal), refusal
    else:
        raise AssertionError("MSAL redeemed an auth response whose state was forged")
    declined = web_app.handle_auth_response(
        dict(flow), {"error": "access_denied", "error_description": "The user declined.", "state": flow["state"]})
    assert declined == {"error": "access_denied", "error_description": "The user declined."}, declined
    assert len(web_authority.requests) == requests_before_reply, web_authority.requests
    
    # The genuine reply: MSAL redeems the code with the PKCE verifier and the app's secret. This authority refuses the
    # unregistered secret (AADSTS7000215), as the real one would, so the result is MSAL's refusal dict, never a token.
    result = web_app.handle_auth_response(flow, {"code": "fake-auth-code", "state": flow["state"]})
    assert result.get("error") == "invalid_client" and "access_token" not in result, result
    token_posts = [request for request in web_authority.requests if request[0] == "POST"]
    assert len(token_posts) == 1, web_authority.requests
    _, token_url, _, token_form = token_posts[0]
    assert token_url == "https://login.microsoftonline.com/common/oauth2/v2.0/token", token_url
    assert token_form["grant_type"] == "authorization_code" and token_form["code"] == "fake-auth-code", token_form
    assert token_form["code_verifier"] == flow["code_verifier"], token_form
    assert token_form["redirect_uri"] == "https://your-app.com/auth/callback", token_form
    assert (token_form["client_id"], token_form["client_secret"]) == ("web-app-client-id", "web-app-client-secret"), token_form
    # Nothing was granted, so the cache never changed and nothing was persisted.
    assert not os.path.exists(web_config["token_cache_file"]), web_config["token_cache_file"]
    print(f"     Sign-in code exchange refused as expected ({result['error']})")
    
    # 2. Daemon/Service pattern
    print("  2. Daemon/Service Pattern:")
    
    class ProductionDaemon:
        """Production daemon service with client credentials flow"""
        
        def __init__(self, config):
            self.config = config
            self.msal_app = msal.ConfidentialClientApplication(
                client_id=config["client_id"],
                client_credential=config["client_credential"],
                authority=config["authority"],
                http_client=config["http_client"]
            )
            
            # Token management
            self.current_token = None
            self.token_expires_at = 0
            self.token_lock = threading.Lock()
        
        def get_access_token(self, scopes=None):
            """Get access token with automatic refresh"""
            if not scopes:
                scopes = [f"https://graph.microsoft.com/.default"]
            
            with self.token_lock:
                # Check if current token is still valid
                if self.current_token and time.time() < self.token_expires_at - 300:  # 5-minute buffer
                    return self.current_token
                
                # Acquire new token
                result = self.msal_app.acquire_token_for_client(scopes=scopes)
                
                if "access_token" in result:
                    self.current_token = result["access_token"]
                    self.token_expires_at = time.time() + result.get("expires_in", 3600)
                    return self.current_token
                else:
                    raise TokenAcquisitionError(result)
        
        def make_graph_api_call(self, endpoint, method="GET", data=None):
            """Make authenticated Graph API call"""
            token = self.get_access_token()
            
            # Simulate API call
            return {
                "status": "success",
                "endpoint": endpoint,
                "method": method,
                "token_preview": token[:20] + "..." if token else None
            }
    
    # Sample daemon configuration
    # A SPECIFIC tenant: MSAL warns that /common and /organizations are unreliable for the client-credentials flow.
    daemon_config = {
        "client_id": "daemon-client-id",
        "client_credential": "daemon-client-secret",  # Or certificate
        "authority": "https://login.microsoftonline.com/contoso.onmicrosoft.com",
        "http_client": FakeAuthority()
    }
    
    daemon_service = ProductionDaemon(daemon_config)
    print(f"     Daemon service initialized for unattended operation")
    
    # Test daemon functionality. The secret is fake, so the authority REFUSES it (AADSTS7000215); MSAL surfaces that as
    # its documented failure result and the daemon raises its own TokenAcquisitionError — never a token.
    try:
        daemon_service.get_access_token()
    except TokenAcquisitionError as refusal:
        assert refusal.result["error"] == "invalid_client" and refusal.result["error_codes"] == [7000215], refusal.result
        assert "access_token" not in refusal.result and daemon_service.current_token is None
        print(f"     Daemon token request refused as expected ({refusal.result['error']})")
    else:
        raise AssertionError("the authority issued a token for a fake client secret")
    
    # The request MSAL formed: one client-credentials POST to the tenant's token endpoint, carrying the app's identity.
    token_posts = [request for request in daemon_config["http_client"].requests if request[0] == "POST"]
    assert len(token_posts) == 1, daemon_config["http_client"].requests
    _, token_url, _, token_form = token_posts[0]
    assert token_url == "https://login.microsoftonline.com/contoso.onmicrosoft.com/oauth2/v2.0/token", token_url
    assert token_form["grant_type"] == "client_credentials", token_form
    assert (token_form["client_id"], token_form["client_secret"]) == ("daemon-client-id", "daemon-client-secret"), token_form
    assert token_form["scope"] == "https://graph.microsoft.com/.default", token_form
    
    # The same daemon once its secret is registered: the authority issues a bearer token, the daemon serves the next
    # call from its own cache (no second POST), and the Graph call carries it.
    class GrantingAuthority(FakeAuthority):
        def post(self, url, params=None, data=None, headers=None, **kwargs):
            self.requests.append(("POST", url, params, data))
            return FakeAuthorityResponse(200, {"token_type": "Bearer", "expires_in": 3599,
                                               "access_token": "eyJ0eXAiOiJKV1Qi.fake-daemon-token"})
    
    granted_daemon = ProductionDaemon(dict(daemon_config, http_client=GrantingAuthority()))
    token = granted_daemon.get_access_token()
    assert token == "eyJ0eXAiOiJKV1Qi.fake-daemon-token", token
    assert granted_daemon.get_access_token() == token
    assert sum(1 for request in granted_daemon.config["http_client"].requests if request[0] == "POST") == 1
    print(f"     Daemon token acquired: {token[:20]}...")
    
    api_result = granted_daemon.make_graph_api_call("/users")
    assert api_result == {"status": "success", "endpoint": "/users", "method": "GET", "token_preview": token[:20] + "..."}, api_result
    print(f"     API call result: {api_result['status']}")
    
    # 3. Multi-tenant application pattern
    print("  3. Multi-tenant Application Pattern:")
    
    class MultiTenantApp:
        """Multi-tenant application with tenant-specific token management"""
        
        def __init__(self, client_id, client_secret, http_client):
            self.client_id = client_id
            self.client_secret = client_secret
            self.http_client = http_client
            self.tenant_apps = {}  # Cache MSAL apps per tenant
        
        def get_msal_app(self, tenant_id):
            """Get or create MSAL app for specific tenant"""
            if tenant_id not in self.tenant_apps:
                authority = f"https://login.microsoftonline.com/{tenant_id}"
                self.tenant_apps[tenant_id] = msal.ConfidentialClientApplication(
                    client_id=self.client_id,
                    client_credential=self.client_secret,
                    authority=authority,
                    http_client=self.http_client
                )
            
            return self.tenant_apps[tenant_id]
        
        def authenticate_user(self, tenant_id, scopes, redirect_uri):
            """Start a sign-in against a specific tenant; the caller keeps the returned flow for the reply.
            (initiate_auth_code_flow replaces the deprecated get_authorization_request_url.)"""
            app = self.get_msal_app(tenant_id)
            flow = app.initiate_auth_code_flow(
                scopes,
                redirect_uri=redirect_uri,
                response_mode="form_post"
            )
            
            return {
                "tenant_id": tenant_id,
                "flow": flow,
                "auth_url": flow["auth_uri"],
                "app_instance": type(app).__name__
            }
    
    # Test multi-tenant pattern
    multi_tenant = MultiTenantApp("multi-tenant-client-id", "client-secret", http_client=FakeAuthority())
    
    # The three multi-tenant endpoints Microsoft ships: each gets its OWN app, its own discovery, its own authorize URL.
    tenant_ids = ["common", "organizations", "consumers"]
    flow_states = []
    for tenant_id in tenant_ids:
        auth_info = multi_tenant.authenticate_user(
            tenant_id=tenant_id,
            scopes=["User.Read"],
            redirect_uri="https://app.com/callback"
        )
        assert auth_info["auth_url"].startswith(f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/authorize?"), auth_info
        assert urllib.parse.parse_qs(urllib.parse.urlsplit(auth_info["auth_url"]).query)["response_mode"] == ["form_post"], auth_info
        flow_states.append(auth_info["flow"]["state"])
        assert auth_info["app_instance"] == "ConfidentialClientApplication", auth_info
        print(f"     Tenant {tenant_id}: Auth URL generated for {auth_info['app_instance']}")
    assert [request[1] for request in multi_tenant.http_client.requests] == [
        f"https://login.microsoftonline.com/{tenant_id}/v2.0/.well-known/openid-configuration" for tenant_id in tenant_ids
    ], multi_tenant.http_client.requests
    assert len(multi_tenant.tenant_apps) == len(tenant_ids)
    # Each sign-in carries its own state, so one tenant's reply can never be redeemed against another's flow.
    assert len(set(flow_states)) == len(tenant_ids), flow_states
    
    # Security best practices
    print("Security Best Practices:")
    
    security_practices = {
        "Token Storage": [
            "Use secure token cache with encryption at rest",
            "Implement token cache isolation per user/tenant",
            "Set appropriate token cache expiration policies",
            "Use database or Redis for distributed cache scenarios"
        ],
        "Certificate Authentication": [
            "Use certificate-based authentication over client secrets",
            "Store certificates in Azure Key Vault or HSM",
            "Implement certificate rotation procedures",
            "Monitor certificate expiration dates"
        ],
        "Network Security": [
            "Use HTTPS only for all authentication flows",
            "Implement proper CORS policies for web applications",
            "Use secure redirect URIs (no wildcards in production)",
            "Implement certificate pinning for mobile apps"
        ],
        "Error Handling": [
            "Implement comprehensive error logging without exposing tokens",
            "Use structured logging for authentication events",
            "Implement retry logic with exponential backoff",
            "Monitor failed authentication attempts"
        ]
    }
    
    for category, practices in security_practices.items():
        print(f"  {category}:")
        for practice in practices:
            print(f"    • {practice}")
    
    # Monitoring and telemetry
    print("Monitoring and Telemetry:")
    
    class MSALTelemetry:
        """MSAL telemetry and monitoring"""
        
        def __init__(self):
            self.events = []
        
        def log_authentication_event(self, event_type, details):
            """Log authentication events for monitoring"""
            event = {
                "timestamp": time.time(),
                "event_type": event_type,
                "details": details,
                "session_id": "session-123"  # Would be actual session ID
            }
            self.events.append(event)
        
        def get_metrics(self):
            """Generate authentication metrics"""
            if not self.events:
                return {"total_events": 0}
            
            event_types = {}
            for event in self.events:
                event_type = event["event_type"]
                event_types[event_type] = event_types.get(event_type, 0) + 1
            
            return {
                "total_events": len(self.events),
                "event_distribution": event_types,
                "time_range": {
                    "start": min(e["timestamp"] for e in self.events),
                    "end": max(e["timestamp"] for e in self.events)
                }
            }
    
    # Test telemetry
    telemetry = MSALTelemetry()
    
    # Log sample events
    sample_events = [
        ("authentication_started", {"method": "interactive", "scopes": ["User.Read"]}),
        ("token_acquired", {"expires_in": 3600, "scope_count": 1}),
        ("token_refreshed", {"previous_expiry": time.time() - 100}),
        ("authentication_error", {"error": "consent_required", "retry_count": 1})
    ]
    
    for event_type, details in sample_events:
        telemetry.log_authentication_event(event_type, details)
    
    metrics = telemetry.get_metrics()
    print(f"  Telemetry metrics:")
    print(f"    Total events: {metrics['total_events']}")
    print(f"    Event types: {list(metrics['event_distribution'].keys())}")
    
    # Performance optimization
    print("Performance Optimization:")
    
    optimization_strategies = [
        "Token caching: Implement efficient token caching to reduce authentication requests",
        "Connection pooling: Use connection pools for Graph API calls",
        "Batch operations: Group multiple Graph API calls into batch requests",
        "Async operations: Use async/await for I/O bound authentication operations", 
        "Load balancing: Distribute authentication load across multiple app instances",
        "CDN integration: Use CDN for static authentication assets",
        "Monitoring: Implement performance monitoring and alerting",
        "Scaling: Design for horizontal scaling in cloud environments"
    ]
    
    for strategy in optimization_strategies:
        print(f"  • {strategy}")
    
    # Compliance and governance
    print("Compliance and Governance:")
    
    compliance_areas = [
        "Data residency: Ensure token storage complies with data residency requirements",
        "Audit logging: Implement comprehensive audit trails for all authentication events", 
        "Access reviews: Regular reviews of application permissions and consents",
        "Conditional access: Integration with Azure AD conditional access policies",
        "MFA enforcement: Support for multi-factor authentication requirements",
        "Privacy controls: Implement privacy controls for user data access",
        "Incident response: Prepare incident response procedures for security events",
        "Compliance reporting: Generate reports for compliance and audit purposes"
    ]
    
    for area in compliance_areas:
        print(f"  • {area}")
    
    print("msal example 5 completed successfully")
    
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"msal example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")